In [ ]:
import os
# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
# os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ['JAX_PLATFORMS'] = 'cpu'

%load_ext autoreload
%autoreload 2

In [ ]:
# test_projection.py
from agent_system.environments.env_package.craftext import craftext_projection, INVALID_ACTION_ID

# Список тестовых выводов от "LLM"
test_actions = [
    "<think>I should go left.</think><action>left</action>",             # Идеальный случай
    "<think>I will try to place a table.</think><action>place table</action>", # Другой идеальный случай
    "<action>up</action>",                                               # Случай без <think>
    "<think>Let's move right.</think>",                                   # Случай без <action>
    "<think>I am confused.</think><action>fly to the moon</action>",    # Невалидное действие
    "<think>你好</think><action>down</action>",                           # Случай с мусором (китайский)
]

processed_actions, valids = craftext_projection(test_actions)

print(f"Processed Actions: {processed_actions}")
print(f"Valids: {valids}")

# Проверки (ассерты)
assert processed_actions == [1, 8, INVALID_ACTION_ID, INVALID_ACTION_ID, INVALID_ACTION_ID, INVALID_ACTION_ID]
assert valids == [1, 1, 0, 0, 0, 0]

print("\nТест проекции прошел успешно!")

In [ ]:
from agent_system.environments.env_package.craftext import CraftextWorker

print("Инициализация воркера...")
# Инициализируем воркер напрямую, как обычный Python-объект
worker = CraftextWorker(seed=42, env_kwargs={})

print("\nТестируем reset()...")
obs, info = worker.reset(scenario_idx=5)

print(f"  - Тип observation: {type(obs)}")
print(f"  - Форма observation: {obs.shape}")
print(f"  - Ключи в info: {info.keys()}")
assert 'text_render' in info
assert 'instruction' in info
print("  - Инструкция:", info['instruction'][:50] + "...") # Печатаем начало инструкции
print("  - Текстовый рендер:", info['text_render'][:70] + "...") # Печатаем начало рендера

print("\nТестируем step()...")
# Действие "RIGHT" (ID=2)
obs, reward, done, info = worker.step(action=2)

print(f"  - Тип reward: {type(reward)}")
print(f"  - Тип done: {type(done)}")
print(f"  - Ключи в info: {info.keys()}")
assert 'text_render' in info
assert 'instruction' in info
print(f"  - Награда: {reward}, Завершено: {done}")

print("\nТесты воркера прошли успешно!")

In [ ]:
import ray

from agent_system.environments.env_package.craftext import build_craftext_envs

ray.shutdown()

ray.init()

print("Создаем векторизованную среду...")
# Используем маленькие значения для теста
env_num = 2
group_n = 2
total_workers = env_num * group_n

env = build_craftext_envs(
    seed=0,
    env_num=env_num,
    group_n=group_n,
    resources_per_worker={'num_cpus': 1},
    is_train=True,
)

print(f"\nТестируем reset() на {total_workers} воркерах...")
obs_list, info_list = env.reset()

print(f"  - Длина списка obs: {len(obs_list)}")
print(f"  - Длина списка info: {len(info_list)}")
assert len(obs_list) == total_workers
assert len(info_list) == total_workers
print("  - Ключи в первом info:", info_list[0].keys())

print(f"\nТестируем step() на {total_workers} воркерах...")
# Отправляем 4 случайных действия
actions = [1, 2, 3, 4]
obs_list, reward_list, done_list, info_list = env.step(actions)

print(f"  - Длина списка rewards: {len(reward_list)}")
assert len(reward_list) == total_workers
print("  - Награды:", reward_list)

print("\nЗакрываем среду...")
env.close()
ray.shutdown()

print("\nИнтеграционный тест прошел успешно!")

In [ ]:
ray.shutdown()

In [ ]:
# profile_worker.py
import time
import os
import psutil # Установите, если нет: pip install psutil
from agent_system.environments.env_package.craftext import CraftextWorker

# Функция для отслеживания памяти
def print_memory_usage(process):
    mem_info = process.memory_info()
    print(f"RAM Usage: {mem_info.rss / 1024 / 1024:.2f} MB")

# --- Основной код ---
worker = CraftextWorker(seed=0, env_kwargs={})
process = psutil.Process(os.getpid())

print("--- Initial Memory ---")
print_memory_usage(process)

print("\n--- Running 10 Episodes ---")
for i in range(10):
    print(f"\nEpisode {i+1}")
    obs, info = worker.reset(scenario_idx=0)
    done = False
    step = 0
    while not done and step < 100: # Ограничим длину эпизода для теста
        obs, reward, done, info = worker.step(action=2) # Просто идем вправо
        step += 1
    
    print(f"Episode finished. Current memory:")
    print_memory_usage(process)
    time.sleep(1)

print("\n--- Profiling Finished ---")

In [ ]:
obs, reward, done, info = worker.step(action=2) # Просто идем вправо